# 네이버 쇼핑 "결혼답례품"검색 시 상위 1페이지 모든 스마트스토어 링크 저장

In [ ]:
## selenium에서 특정 기능을 로드
# By : 특정 태그를 찾을 때 ID나 Class, ... 조건으로 검색
# 라이브러리 불러오기
from selenium import webdriver 
from selenium.webdriver.common.by import By
# Keys : 키보드의 이벤트를 발생 시키는 기능
from selenium.webdriver.common.keys import Keys
import time
# 웹 브라우저를 실행
driver = webdriver.Chrome()

# 웹 브라우저에 주소를 입력
driver.get('https://search.shopping.naver.com/search/all?query=%EA%B2%B0%ED%98%BC%EB%8B%B5%EB%A1%80%ED%92%88&bt=-1&frm=NVSCVUI')


### 네이버 쇼핑 웹브라우저 스크롤을 내려 모든 스마트스토어 로드

In [3]:
SCROLL_PAUSE_SEC = 2

# 스크롤 높이 가져옴
last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    # 끝까지 스크롤 다운
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # 1초 대기
    time.sleep(SCROLL_PAUSE_SEC)

    # 스크롤 다운 후 스크롤 높이 다시 가져옴
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

### 1페이지 모든 스마트 스토어 링크 저장

In [4]:
# 클래스 이름이 'thumbnail_thumb__Bxb6Z'인 요소들을 찾기
elements = driver.find_elements(By.CSS_SELECTOR, '.thumbnail_thumb__Bxb6Z.linkAnchor')
links=[]
# 찾은 요소들 출력
for element in elements:
    try:
        link = element.get_attribute('href')
        links.append(link)
    except:
        continue
len(links)


44

### 가져온 모든 스마트스토어링크에 하나씩 접속하여 리뷰 20페이지씩 모두 수집

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import re

time_deplay = 0.5

# 링크 리스트 (예시)

driver = webdriver.Chrome()

# 리뷰를 저장할 리스트 초기화
all_reviews = []

# 각 링크에 대해 리뷰를 수집
for link in links:
    # 웹 브라우저 실행 및 링크로 이동
    driver.get(link)
    time.sleep(3)

    # 페이지 스크롤을 위해 body 태그 선택
    body = driver.find_element(By.TAG_NAME, 'body')

    SCROLL_PAUSE_SEC = 2
    # 스크롤 높이 가져옴
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        # 끝까지 스크롤 다운
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # 1초 대기
        time.sleep(SCROLL_PAUSE_SEC)

        # 스크롤 다운 후 스크롤 높이 다시 가져옴
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
    time.sleep(2)


    # 리뷰 버튼 클릭
    review_btn = driver.find_element(By.CSS_SELECTOR,'#_productFloatingTab > div > div._27jmWaPaKy._1dDHKD1iiX > ul > li:nth-child(2) > a')
    time.sleep(2)
    review_btn.click()
    time.sleep(2)
    # 스크롤 높이 가져옴
    last_height = driver.execute_script("return document.body.scrollHeight")
    # 원하는 요소를 찾기 (CSS 선택자 사용)
    element = driver.find_element(By.CSS_SELECTOR, 'a.fAUKm1ewwo._2Ar8-aEUTq._nlog_click[data-shp-page-key="100329229"]')
    # 요소로 스크롤 이동
    driver.execute_script("arguments[0].scrollIntoView(false);", element)
    time.sleep(time_deplay)


    page_count = 0
    max_pages = 20

    while page_count < max_pages:
        time.sleep(time_deplay)
        ul_element = driver.find_element(By.CSS_SELECTOR, "ul[class='_2ms2i3dD92']")
        elements = ul_element.find_elements(By.CSS_SELECTOR, 'li')
        print(len(elements))
        time.sleep(time_deplay)
        for element in elements:
            try:
                element.find_element(By.CSS_SELECTOR, 'a[class="_3qc99-a3Vq"]').click()
            except:
                print('다음으로 이동')
            try:
                review = element.find_elements(By.CSS_SELECTOR, "span[class='_2L3vDiadT9']")[1].text
                review = re.sub('[^#0-9a-zA-Zㄱ-힣 ]', "", review)  # 불필요한 문자 제거
                print(review)
                all_reviews.append(review)

            except Exception as e:
                print(e)
        try:
            # 다음 페이지 버튼이 클릭 가능할 때까지 기다림
            next_button = driver.find_element(By.CSS_SELECTOR, 'a.fAUKm1ewwo._2Ar8-aEUTq._nlog_click[data-shp-page-key="100329229"]')
            next_button.click()
            time.sleep(time_deplay)
            driver.execute_script("arguments[0].scrollIntoView(false);", next_button)
            page_count += 1  # 페이지 카운터 증가
        except Exception as e:
            print(f"Error while clicking next button: {e}")
            break

# 브라우저 닫기
driver.quit()



### 리뷰가 잘 수집되었는지 확인

In [23]:
all_reviews

['크기는 작지만 포장ㆍ구성이 고급져요 받으시는분들이 넘좋아하셔서 기분이 좋았어요 쇼핑백이 왜비싼가했는데 튼튼하고 좋은거여서 선택하길 잘한거같아요 답례품으로 아주 성공적이었어요 배송도 잘해주시고 꼼꼼한검수 부탁드렸는데 불량한개도없어서 만족했습니다 스티커문구제작 서비스 넘았어요',
 '정말 받기전까진 잘 몰랐는데 너무 예쁘고 답례품으로 하기 좋네요 추천합니다배송전에 이것저것 고민하고 문의도 많이 했는데 친절하게 다 답변해주시고 배송도 하루만에 왔네요진짜 만족합니다',
 '가족식사자리 답례품으로 사용하려고 미리 주문하였습니다 포장도 꼼꼼하고 디자인도이쁘고 상담도 친절하게 해주셨습니다다음에 또 주문할께요쇼핑백추가 너무 잘한것같아요감사합니다',
 '이사가면서 공사소음으로 고생하시는 이웃분들께 돌리려고 샀어요 판매자님 매우 빠르게 보내주셔서 잘 전달할 수 있었습니다 쇼핑백은 여기 판매하지 않아서 저희가 다이소가서 어울리는걸로 따로 구매했어요',
 '두찌 돌잔치에 잘 썼어요인원을 몇안해서 50개만 맞췄는데봉투만하고 못오신분들까지 드리니몇개안남아요ㅎㅎ넉넉히 주문 잘한듯요깨진거 있나없나 확인하려고 집으로 배송시켰는데 포장도 꼼꼼하고깨진거 없이 잘 와서바로 행사장에 가져다 두고 행사잘 치르고 답례품도 잘 드리고잘 마쳤습니다답례품이고급스럽다보니 많이 만족하시더라구요감사드려요',
 '결혼 답례품으로 돌렸습니다 살짝 비싼 것 같기도 하지만 받으신 분들이 전부 고급스럽다고 좋아해주셔서 뿌듯했습니다 감사합니다 사업 번창하세요ㅎㅎ',
 '결혼 답례품으로 돌렸습니다 살짝 비싼 것 같기도 하지만 받으신 분들이 전부 고급스럽다고 좋아해주셔서 뿌듯했습니다 감사합니다 사업 번창하세요ㅎㅎ',
 '박스에 딱 맞게 포장해주셔서제품 손상없이 물건 잘 받았습니다좋습니다 담례품으로 너무 좋은 것 같아요',
 '디자인 신경 쓴 듯한 포장과  답례 스티커가 답례품으로 하기 정말 좋게 나왔어요결혼식 후 양가 지인에게 총 200개 돌렸는데 다들 가격 대비 비싸게 봐주시더라구여무게감이 있다보니 답례가 고급져 보입니

In [24]:
len(all_reviews)

12371

#### 수집한 리뷰 데이터프레임 변환 및 csv파일 저장

In [25]:
# 데이터프레임으로 변환
df = pd.DataFrame(all_reviews, columns=['Review'])

# 데이터프레임 출력 및 저장
print(df)
df.to_csv('all_reviews.csv', index=False, encoding='utf-8-sig')

                                                  Review
0      크기는 작지만 포장ㆍ구성이 고급져요 받으시는분들이 넘좋아하셔서 기분이 좋았어요 쇼핑...
1      정말 받기전까진 잘 몰랐는데 너무 예쁘고 답례품으로 하기 좋네요 추천합니다배송전에 ...
2      가족식사자리 답례품으로 사용하려고 미리 주문하였습니다 포장도 꼼꼼하고 디자인도이쁘고...
3      이사가면서 공사소음으로 고생하시는 이웃분들께 돌리려고 샀어요 판매자님 매우 빠르게 ...
4      두찌 돌잔치에 잘 썼어요인원을 몇안해서 50개만 맞췄는데봉투만하고 못오신분들까지 드...
...                                                  ...
12366                           떡 돌리는거 보다 훨 간편하고 좋은거 같아요
12367                                      포장 아주 이쁘고 좋아요
12368                                      이쁘고 좋아요 추천합니다
12369  요구사항이 좀 있었는데 카톡도 친절하게 답변해주시고 배송도 엄청 빨라요퇴사하면서 직...
12370  학원선생님들께 드렸더니 인기폭발한 답례품 다들 꿀도 맛있고 요즘 추워서 환절기에 좋...

[12371 rows x 1 columns]
